###Setup: Google Drive and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import json
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple
from collections import Counter, defaultdict
from dataclasses import dataclass
import subprocess

try:
    from rapidfuzz import fuzz
except ImportError:
    print("Installing rapidfuzz for fuzzy matching...")
    os.system("pip install rapidfuzz -q")
    from rapidfuzz import fuzz

import math

###LLaMA Factory Setup

In [ ]:
!apt-get update
!apt-get install -y libavformat-dev libavcodec-dev libavdevice-dev \
    libavutil-dev libswscale-dev libswresample-dev libavfilter-dev pkg-config

# Install stable Torch 2.8 (avoid Torch 2.9 Conv3D bug)
!pip install --force-reinstall torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 \
    --index-url https://download.pytorch.org/whl/cu126

!pip install -U bitsandbytes

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,315 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,664 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.5 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,205 kB]
Get:13 https://cloud.r-project.org/bin/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 43.9 MB/s eta 0:00:00


In [ ]:
# Clone and install LLaMA Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]"

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 526, done.
remote: Counting objects: 100% (526/526), done.
remote: Compressing objects: 100% (408/408), done.
remote: Total 526 (delta 122), reused 316 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (526/526), 5.21 MiB | 12.95 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/LLaMA-Factory
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━

###Dataset Info

In [ ]:
def generate_test_subset_dataset_info(
    subset_size: int,
    model: str = "qwen",
    prompt_type: str = "standard",
    base_drive_path: str = "/content/drive/MyDrive/NLP/Thesis"
) -> Dict[str, Any]:
    """
    Generate dataset_info entries for a test subset.
    """
    type_suffix = "strict"

    prefix = f"hotel50k_{model}"

    base_path = f"{base_drive_path}/llama_factory_data/{model}/json_datasets_{type_suffix}"

    entries = {}
    for stage in [1, 2, 3, 4]:
        key = f"{prefix}_test_subset_{subset_size}_s{stage}"
        entries[key] = {
            "file_name": f"{base_path}/test_subset_{subset_size}/test_stage{stage}.json",
            "formatting": "alpaca",
            "columns": {
                "prompt": "instruction",
                "query": "input",
                "response": "output",
                "images": "images"
            }
        }

    return entries

In [ ]:
def generate_all_dataset_info(
    base_drive_path: str = "/content/drive/MyDrive/NLP/Thesis",
    test_subset_sizes: List[int] = None
) -> Dict[str, Any]:
    """
    Generate all dataset_info entries for training and evaluation.
    """
    if test_subset_sizes is None:
        test_subset_sizes = []

    all_entries = {}
    sizes = ["1000", "5000", "10000", "100000", "500000", "full"]
    stages = [1, 2, 3, 4]

    # Model and prompt type combinations
    configs = [
        ("qwen", "standard", "hotel50k_qwen", "strict"),
        ("gemma", "standard", "hotel50k_gemma", "strict"),
    ]

    for model, prompt_type, prefix, type_suffix in configs:
        base_path = f"{base_drive_path}/llama_factory_data/{model}/json_datasets_{type_suffix}"

        # Training and validation datasets
        for size in sizes:
            for stage in stages:
                # Train
                all_entries[f"{prefix}_{size}_train_s{stage}"] = {
                    "file_name": f"{base_path}/{size}/train_stage{stage}.json",
                    "formatting": "alpaca",
                    "columns": {
                        "prompt": "instruction",
                        "query": "input",
                        "response": "output",
                        "images": "images"
                    }
                }
                # Val
                all_entries[f"{prefix}_{size}_val_s{stage}"] = {
                    "file_name": f"{base_path}/{size}/val_stage{stage}.json",
                    "formatting": "alpaca",
                    "columns": {
                        "prompt": "instruction",
                        "query": "input",
                        "response": "output",
                        "images": "images"
                    }
                }

        # Full test set
        for stage in stages:
            all_entries[f"{prefix}_test_s{stage}"] = {
                "file_name": f"{base_path}/test_full/test_stage{stage}.json",
                "formatting": "alpaca",
                "columns": {
                    "prompt": "instruction",
                    "query": "input",
                    "response": "output",
                    "images": "images"
                }
            }

        # Test subsets
        for subset_size in test_subset_sizes:
            for stage in stages:
                all_entries[f"{prefix}_test_subset_{subset_size}_s{stage}"] = {
                    "file_name": f"{base_path}/test_subset_{subset_size}/test_stage{stage}.json",
                    "formatting": "alpaca",
                    "columns": {
                        "prompt": "instruction",
                        "query": "input",
                        "response": "output",
                        "images": "images"
                    }
                }

    return all_entries

In [ ]:
def merge_dataset_info(
    llama_factory_path: str = "/content/LLaMA-Factory",
    base_drive_path: str = "/content/drive/MyDrive/NLP/Thesis",
    test_subset_sizes: List[int] = None
):

    dataset_info_path = os.path.join(llama_factory_path, "data", "dataset_info.json")

    # Load existing data, generate new entries and merge
    with open(dataset_info_path, 'r') as f:
        existing = json.load(f)

    new_entries = generate_all_dataset_info(base_drive_path, test_subset_sizes)
    existing.update(new_entries)

    with open(dataset_info_path, 'w') as f:
        json.dump(existing, f, indent=2)

    print(f"Merged {len(new_entries)} dataset entries into {dataset_info_path}")

    # Print summary
    qwen_count = sum(1 for k in new_entries if "qwen" in k)
    gemma_count = sum(1 for k in new_entries if "gemma" in k)
    print(f"  Qwen entries: {qwen_count}")
    print(f"  Gemma entries: {gemma_count}")

###Config

In [ ]:
@dataclass
class EvalConfig:
    """
    Configuration for evaluation runs.
    """
    base_drive_path: str = "/content/drive/MyDrive/NLP/Thesis"
    yaml_dir: str = "/content/drive/MyDrive/NLP/Thesis/yaml-eval-files"
    checkpoint_base: str = "/content/drive/MyDrive/NLP/Thesis/llama_factory_data/checkpoints"
    results_base: str = "/content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results"
    llama_factory_path: str = "/content/LLaMA-Factory"

    # Dataset paths for different model/prompt combinations
    dataset_base: str = "/content/drive/MyDrive/NLP/Thesis/llama_factory_data"

    def __post_init__(self):
        """Create necessary directories."""
        os.makedirs(self.yaml_dir, exist_ok=True)
        os.makedirs(self.results_base, exist_ok=True)


# Model configurations
MODEL_CONFIGS = {
    "qwen3vl-4b": {
        "model_path": "Qwen/Qwen3-VL-4B-Instruct",
        "template": "qwen3_vl_nothink",
        "dataset_prefix": "hotel50k_qwen",
    },
    "gemma-3-4b-it": {
        "model_path": "google/gemma-3-4b-it",
        "template": "gemma3",
        "dataset_prefix": "hotel50k_gemma",
    },
}

# GPU configurations for batch sizes
GPU_BATCH_SIZES = {
    "l4": 32,
    "a100_40gb": 64,
    "a100_80gb": 128,
}

STAGES = [1, 2, 3, 4]
SIZES = ["1000", "5000", "10000", "100000", "500000", "full"]

# Evaluation modes for top-k accuracy
EVAL_MODES = {
    "top1": {
        "suffix": "",
        "k": 1,
        "description": "Standard single prediction"
    },
    # Top3, top5 prompts were ultimately not used. Additionally the implementation was incorrect.
    # For future implementations, the response should be an array of predictions.

}

###Yaml Files

In [ ]:
def generate_base_model_eval_yaml(
    config: EvalConfig,
    model_name: str = "qwen3vl-4b",
    stage: int = 1,
    test_dataset: str = None,
    batch_size: int = 128,
    eval_mode: str = "top1"
) -> str:
    """
    Generate evaluation YAML for base (non-fine-tuned) model.
    """
    model_config = MODEL_CONFIGS.get(model_name)
    if not model_config:
        raise ValueError(f"Unknown model: {model_name}. Available: {list(MODEL_CONFIGS.keys())}")

    # Determine dataset name and output path
    if test_dataset is None:
        test_dataset = f"{model_config['dataset_prefix']}_test_s{stage}"

    output_path = os.path.join(
        config.results_base,
        f"{model_name}/{eval_mode}/base_model/stage_{stage}"
    )

    yaml_content = f"""### Model Configuration
model_name_or_path: {model_config['model_path']}
template: {model_config['template']}
trust_remote_code: true

### Evaluation Method
stage: sft
do_predict: true

### Dataset Configuration
eval_dataset: {test_dataset}
cutoff_len: 4096
max_new_tokens: 512
overwrite_cache: true

### Inference Parameters
per_device_eval_batch_size: {batch_size}
predict_with_generate: true
temperature: 0.1
top_p: 0.9

### Output & Logging
output_dir: {output_path}
report_to: none
overwrite_output_dir: true
"""

    yaml_filename = f"{model_name}_base_eval_s{stage}.yaml"
    yaml_path = os.path.join(config.yaml_dir, yaml_filename)

    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    print(f"Generated: {yaml_path}")
    return yaml_path

In [ ]:
def generate_finetuned_model_eval_yaml(
    config: EvalConfig,
    model_name: str = "qwen3vl-4b",
    gpu: str = "l4",
    dataset_size: str = "10000",
    stage: int = 1,
    test_dataset: str = None,
    batch_size: int = None,
    eval_mode: str = "top1"
) -> str:
    """
    Generate evaluation YAML for a fine-tuned model.
    """
    model_config = MODEL_CONFIGS.get(model_name)
    if not model_config:
        raise ValueError(f"Unknown model: {model_name}. Available: {list(MODEL_CONFIGS.keys())}")

    checkpoint_path = os.path.join(
        config.checkpoint_base,
        model_name,
        gpu,
        f"{dataset_size}_stage{stage}"
    )

    if not os.path.exists(checkpoint_path):
        print(f"Warning: Checkpoint not found at {checkpoint_path}")

    if test_dataset is None:
        test_dataset = f"{model_config['dataset_prefix']}_test_s{stage}"

    if batch_size is None:
        batch_size = GPU_BATCH_SIZES.get(gpu, 64)

    output_path = os.path.join(
        config.results_base,
        f"{model_name}/{eval_mode}/{gpu}/{dataset_size}/stage_{stage}"
    )

    yaml_content = f"""### Model Configuration
model_name_or_path: {model_config['model_path']}
adapter_name_or_path: {checkpoint_path}
template: {model_config['template']}
trust_remote_code: true

### Evaluation Method
stage: sft
do_predict: true
finetuning_type: lora

### Dataset Configuration
eval_dataset: {test_dataset}
cutoff_len: 4096
max_new_tokens: 512
overwrite_cache: true

### Inference Parameters
per_device_eval_batch_size: {batch_size}
predict_with_generate: true
temperature: 0.1
top_p: 0.9

### Output & Logging
output_dir: {output_path}
report_to: none
overwrite_output_dir: true
"""

    yaml_filename = f"{model_name}_{gpu}_{dataset_size}_eval_s{stage}.yaml"
    yaml_path = os.path.join(config.yaml_dir, yaml_filename)

    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    print(f"Generated: {yaml_path}")
    return yaml_path

In [ ]:
def generate_all_eval_yamls(
    config: EvalConfig,
    model_name: str = "qwen3vl-4b",
    gpu: str = "l4",
    dataset_sizes: List[str] = None,
    stages: List[int] = None,
    include_base: bool = True,
    test_subset_size: int = None,
    eval_mode: str = "top1"
) -> List[str]:
    """
    Generate all evaluation YAML files for a model/GPU combination.
    """
    if dataset_sizes is None:
        dataset_sizes = SIZES
    if stages is None:
        stages = STAGES

    if eval_mode not in EVAL_MODES:
        raise ValueError(f"eval_mode must be one of {list(EVAL_MODES.keys())}, got: {eval_mode}")

    yaml_paths = []

    # Determine test dataset name
    model_config = MODEL_CONFIGS.get(model_name)
    mode_suffix = EVAL_MODES[eval_mode]["suffix"]

    if test_subset_size:
        # e.g., hotel50k_qwen_test_subset_10000_s{stage} or hotel50k_qwen_test_subset_10000_top3_s{stage}
        if eval_mode == "top1":
            test_dataset_template = f"{model_config['dataset_prefix']}_test_subset_{test_subset_size}_s{{stage}}"
        else:
            test_dataset_template = f"{model_config['dataset_prefix']}_test_subset_{test_subset_size}{mode_suffix}_s{{stage}}"
    else:
        # e.g., hotel50k_qwen_test_s{stage} or hotel50k_qwen_test_top3_s{stage}
        if eval_mode == "top1":
            test_dataset_template = f"{model_config['dataset_prefix']}_test_s{{stage}}"
        else:
            test_dataset_template = f"{model_config['dataset_prefix']}_test{mode_suffix}_s{{stage}}"

    # Generate base model YAMLs
    if include_base:
        print(f" Base Model YAMLs ({model_name})")
        for stage in stages:
            test_dataset = test_dataset_template.format(stage=stage)
            path = generate_base_model_eval_yaml(
                config, model_name, stage, test_dataset=test_dataset, eval_mode=eval_mode
            )
            yaml_paths.append(path)

    # Generate fine-tuned model YAMLs
    print(f"Fine-tuned Model YAMLs ({model_name}, {gpu})")
    for size in dataset_sizes:
        for stage in stages:
            test_dataset = test_dataset_template.format(stage=stage)
            path = generate_finetuned_model_eval_yaml(
                config, model_name, gpu, size, stage, test_dataset=test_dataset, eval_mode=eval_mode
            )
            yaml_paths.append(path)

    print(f"\nGenerated {len(yaml_paths)} YAML files total")

    if test_subset_size:
        print(f"Using test subset: {test_subset_size} samples")

    print(f"Evaluation mode: {eval_mode} ({EVAL_MODES[eval_mode]['description']})")
    return yaml_paths

###Evaluation Functions

In [ ]:
def run_evaluation(yaml_path: str, llama_factory_path: str = "/content/LLaMA-Factory") -> bool:
    """
    Run LLaMA Factory evaluation using a YAML config file.
    """
    print(f"Running evaluation: {os.path.basename(yaml_path)}")

    # Change to LLaMA Factory directory and run
    cmd = f"cd {llama_factory_path} && llamafactory-cli train {yaml_path}"

    try:
        # Use Popen for real-time output streaming
        process = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,  # Merge stderr into stdout
            text=True,
            bufsize=1  # Line buffered
        )

        # Stream output in real-time
        for line in process.stdout:
            print(line, end='')

        process.wait()

        if process.returncode != 0:
            print(f"\nEvaluation failed with return code: {process.returncode}")
            return False

        return True
    except Exception as e:
        print(f"Error running evaluation: {e}")
        return False

In [ ]:
def run_batch_evaluation(
    yaml_paths: List[str],
    llama_factory_path: str = "/content/LLaMA-Factory"
) -> Dict[str, bool]:
    """
    Run multiple evaluations in sequence.
    """
    results = {}

    for i, yaml_path in enumerate(yaml_paths):
        print(f"\n[{i+1}/{len(yaml_paths)}] Processing {os.path.basename(yaml_path)}")
        success = run_evaluation(yaml_path, llama_factory_path)
        results[yaml_path] = success

        if not success:
            print(f"Warning: Evaluation failed for {yaml_path}")

    # Summary
    successful = sum(results.values())
    print(f"Evaluation Summary: {successful}/{len(yaml_paths)} successful")

    return results

###Metrics Calculation

In [ ]:
def normalize_string(s: str) -> str:

    if not s or not isinstance(s, str):
        return ""

    # Lowercase and strip
    # Then remove white spaces and deal with normalizations

    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)

    s = s.replace("united states of america", "united states")
    s = s.replace("u.s.a.", "united states")
    s = s.replace("usa", "united states")
    s = s.replace("uk", "united kingdom")
    s = s.replace("u.k.", "united kingdom")

    return s

In [ ]:
def normalize_json_schema(parsed: Dict) -> Dict:
    """
    Normalize various model output schemas to expected format:
    {city, region, country, coordinates: {latitude, longitude}}
    """
    if not isinstance(parsed, dict):
        return parsed

    result = {}

    # Handle nested wrappers like {"geolocation": {...}}
    wrapper_keys = ('geolocation', 'hotel_room_geolocation', 'location',
                    'result', 'response', 'prediction', 'data')
    if len(parsed) == 1:
        key = list(parsed.keys())[0]
        if key.lower() in wrapper_keys:
            inner = parsed[key]
            if isinstance(inner, dict):
                parsed = inner

    # Extract location fields (try various keys)
    field_alternatives = {
        'country': ['country', 'nation', 'country_name', 'country_code'],
        'region': ['region', 'state', 'province', 'region_name', 'state_name', 'administrative_area'],
        'city': ['city', 'city_name', 'town', 'locality', 'municipality'],
    }

    for field, alternatives in field_alternatives.items():
        for alt in alternatives:
            # Case-insensitive lookup
            for k, v in parsed.items():
                if k.lower() == alt.lower():
                    result[field] = v
                    break
            if field in result:
                break

    # Extract coordinates (try various structures)
    coords = None

    # Direct coordinates field
    if 'coordinates' in parsed and isinstance(parsed['coordinates'], dict):
        coords = parsed['coordinates']
    # Geolocation field (common in base model outputs)
    elif 'geolocation' in parsed and isinstance(parsed['geolocation'], dict):
        coords = parsed['geolocation']
    # Location.coordinates nested
    elif 'location' in parsed and isinstance(parsed['location'], dict):
        loc = parsed['location']
        if 'coordinates' in loc:
            coords = loc['coordinates']
        elif 'latitude' in loc:
            coords = loc
    # Flat lat/lon at top level
    elif 'latitude' in parsed and 'longitude' in parsed:
        coords = {'latitude': parsed['latitude'], 'longitude': parsed['longitude']}
    elif 'lat' in parsed and 'lon' in parsed:
        coords = {'latitude': parsed['lat'], 'longitude': parsed['lon']}
    elif 'lat' in parsed and 'lng' in parsed:
        coords = {'latitude': parsed['lat'], 'longitude': parsed['lng']}

    if coords:
        lat = None
        lon = None

        if isinstance(coords, list) and len(coords) >= 2:
            lat, lon = coords[0], coords[1]
        elif isinstance(coords, dict):
            lat = coords.get('latitude') or coords.get('lat')
            lon = coords.get('longitude') or coords.get('lon') or coords.get('lng')

        if lat is not None and lon is not None:  #
            result['coordinates'] = {'latitude': lat, 'longitude': lon}

    # If we didn't extract standard fields but have other data, preserve it
    if not result:
        return parsed

    # Preserve any additional fields from original that weren't mapped
    for k, v in parsed.items():
        if k not in result and k not in ('geolocation', 'coordinates'):
            result[k] = v

    return result

In [ ]:
def parse_json_response(response: str) -> Optional[Dict]:

    if not response:
        return None

    # Strip think tags
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    # Try to extract JSON from markdown code blocks
    json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response, re.DOTALL)
    if json_match:
        response = json_match.group(1)

    parsed = None

    # Try parsing whole response first
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError:
        pass

    # Find outermost JSON object
    if not parsed:
        start = response.find('{')
        if start != -1:
            depth = 0
            for i, char in enumerate(response[start:], start):
                if char == '{':
                    depth += 1
                elif char == '}':
                    depth -= 1
                    if depth == 0:
                        try:
                            parsed = json.loads(response[start:i+1])
                        except json.JSONDecodeError:
                            pass
                        break

    if not parsed:
        return None

    # Normalize alternative schemas
    return normalize_json_schema(parsed)

In [ ]:
def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:

    R = 6371  # Earth's radius in kilometers

    # Convert to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))

    return R * c

In [ ]:
def calculate_accuracy(
    predictions: List[Dict],
    ground_truth: List[Dict],
    stage: int,
    fuzzy_threshold: int = 85
) -> Dict[str, Any]:
    """
    Calculate accuracy metrics for model predictions.
    """
    if len(predictions) != len(ground_truth):
        print(f"Warning: Prediction count ({len(predictions)}) != ground truth ({len(ground_truth)})")

    # Fields to evaluate per stage
    stage_fields = {
        1: ["country"],
        2: ["region", "country"],
        3: ["city", "region", "country"],
        4: ["city", "region", "country"],  # Coordinates handled separately
    }

    fields = stage_fields.get(stage, ["country"])

    # Initialize counters
    exact_correct = {f: 0 for f in fields}
    fuzzy_correct = {f: 0 for f in fields}
    total = min(len(predictions), len(ground_truth))

    # Per-country accuracy tracking
    country_results = defaultdict(lambda: {"correct": 0, "total": 0})

    # Coordinate distance tracking (Stage 4)
    distances = []
    distance_thresholds = {100: 0, 500: 0, 1000: 0}  # km

    # Detailed results for error analysis
    detailed_results = []

    for i in range(total):
        pred = predictions[i] if predictions[i] else {}
        gt = ground_truth[i] if ground_truth[i] else {}

        result = {"index": i, "prediction": pred, "ground_truth": gt}

        # Get ground truth country for per-country tracking
        gt_country = gt.get("country", "Unknown")
        country_results[gt_country]["total"] += 1

        # Check each field
        field_results = {}
        all_correct = True

        for field in fields:
            pred_val = normalize_string(pred.get(field, ""))
            gt_val = normalize_string(gt.get(field, ""))

            # Exact match
            exact = pred_val == gt_val
            if exact:
                exact_correct[field] += 1

            # Fuzzy match
            fuzzy_score = fuzz.ratio(pred_val, gt_val) if pred_val and gt_val else 0
            fuzzy = fuzzy_score >= fuzzy_threshold
            if fuzzy:
                fuzzy_correct[field] += 1

            field_results[field] = {
                "exact": exact,
                "fuzzy": fuzzy,
                "fuzzy_score": fuzzy_score
            }

            if not fuzzy:
                all_correct = False

        # Track per-country accuracy (based on country field)
        if field_results.get("country", {}).get("fuzzy", False):
            country_results[gt_country]["correct"] += 1

        result["field_results"] = field_results

        # Handle coordinates for Stage 4
        if stage == 4:
            pred_coords = pred.get("coordinates", {})
            gt_coords = gt.get("coordinates", {})

            try:
                # Only calculate distance if both predictions have valid coordinates
                pred_lat = pred_coords.get("latitude") if isinstance(pred_coords, dict) else None
                pred_lon = pred_coords.get("longitude") if isinstance(pred_coords, dict) else None
                gt_lat = gt_coords.get("latitude") if isinstance(gt_coords, dict) else None
                gt_lon = gt_coords.get("longitude") if isinstance(gt_coords, dict) else None

                # Check all coordinates are valid numbers
                if all(v is not None for v in [pred_lat, pred_lon, gt_lat, gt_lon]):
                    pred_lat = float(pred_lat)
                    pred_lon = float(pred_lon)
                    gt_lat = float(gt_lat)
                    gt_lon = float(gt_lon)

                    # Check for (0,0) placeholder - use small tolerance for floating point
                    pred_is_null = (abs(pred_lat) < 0.1 and abs(pred_lon) < 0.1)
                    gt_is_null = (abs(gt_lat) < 0.1 and abs(gt_lon) < 0.1)
                    pred_valid_range = (-90 <= pred_lat <= 90 and -180 <= pred_lon <= 180)
                    gt_valid_range = (-90 <= gt_lat <= 90 and -180 <= gt_lon <= 180)

                    if pred_is_null:
                        result["coordinate_distance_km"] = None
                        result["coordinate_error"] = "Prediction is (0,0) placeholder"
                    elif not pred_valid_range:
                        result["coordinate_distance_km"] = None
                        result["coordinate_error"] = f"Prediction out of range: ({pred_lat}, {pred_lon})"
                    elif gt_is_null or not gt_valid_range:
                        result["coordinate_distance_km"] = None
                        result["coordinate_error"] = "Invalid ground truth coordinates"
                    else:
                        dist = haversine_distance(pred_lat, pred_lon, gt_lat, gt_lon)
                        distances.append(dist)

                        for threshold in distance_thresholds:
                            if dist <= threshold:
                                distance_thresholds[threshold] += 1

                        result["coordinate_distance_km"] = dist
                else:
                    result["coordinate_distance_km"] = None
                    result["coordinate_error"] = "Missing coordinates"
            except (ValueError, TypeError) as e:
                result["coordinate_distance_km"] = None
                result["coordinate_error"] = str(e)

        detailed_results.append(result)

    # Calculate summary metrics
    metrics = {
        "total_samples": total,
        "stage": stage,
        "exact_accuracy": {f: exact_correct[f] / total if total > 0 else 0 for f in fields},
        "fuzzy_accuracy": {f: fuzzy_correct[f] / total if total > 0 else 0 for f in fields},
        "exact_correct_counts": exact_correct,
        "fuzzy_correct_counts": fuzzy_correct,
    }

    # Overall accuracy (all fields correct)
    all_fields_correct = sum(
        1 for r in detailed_results
        if all(r["field_results"].get(f, {}).get("fuzzy", False) for f in fields)
    )
    metrics["overall_accuracy"] = all_fields_correct / total if total > 0 else 0

    # Per-country breakdown
    metrics["per_country_accuracy"] = {
        country: data["correct"] / data["total"] if data["total"] > 0 else 0
        for country, data in country_results.items()
    }
    metrics["per_country_counts"] = dict(country_results)

    # Coordinate metrics for Stage 4
    if stage == 4:
        # Count different error types
        zero_zero_count = sum(
            1 for r in detailed_results
            if r.get("coordinate_error") == "Prediction is (0,0) placeholder"
        )
        missing_count = sum(
            1 for r in detailed_results
            if r.get("coordinate_error") == "Missing coordinates"
        )
        other_invalid = total - len(distances) - zero_zero_count - missing_count

        if distances:
            sorted_distances = sorted(distances)
            metrics["coordinate_metrics"] = {
                "valid_samples": len(distances),
                "zero_zero_predictions": zero_zero_count,
                "missing_coordinates": missing_count,
                "other_invalid": other_invalid,
                "mean_distance_km": sum(distances) / len(distances),
                "median_distance_km": sorted_distances[len(distances) // 2],
                "min_distance_km": sorted_distances[0],
                "max_distance_km": sorted_distances[-1],
                "within_100km": distance_thresholds[100] / len(distances),
                "within_500km": distance_thresholds[500] / len(distances),
                "within_1000km": distance_thresholds[1000] / len(distances),
            }
        else:
            metrics["coordinate_metrics"] = {
                "valid_samples": 0,
                "zero_zero_predictions": zero_zero_count,
                "missing_coordinates": missing_count,
                "other_invalid": other_invalid,
                "error": "No valid coordinate predictions found"
            }

    metrics["detailed_results"] = detailed_results

    return metrics

In [ ]:
def calculate_topk_accuracy(
    predictions: List[Dict],
    ground_truth: List[Dict],
    stage: int,
    k: int = 3,
    fuzzy_threshold: int = 85
) -> Dict[str, Any]:
    """
    Calculate top-k accuracy metrics for model predictions.
    """
    stage_fields = {
        1: ["country"],
        2: ["region", "country"],
        3: ["city", "region", "country"],
        4: ["city", "region", "country"],
    }

    fields = stage_fields.get(stage, ["country"])
    total = min(len(predictions), len(ground_truth))

    # Track accuracy at different k values
    topk_correct = {i: {f: 0 for f in fields} for i in range(1, k + 1)}
    topk_all_correct = {i: 0 for i in range(1, k + 1)}

    detailed_results = []

    for i in range(total):
        pred = predictions[i] if predictions[i] else {}
        gt = ground_truth[i] if ground_truth[i] else {}

        # Extract predictions array
        pred_list = pred.get("predictions", [pred])  # Fallback to single pred
        if not isinstance(pred_list, list):
            pred_list = [pred_list]

        result = {
            "index": i,
            "predictions": pred_list,
            "ground_truth": gt,
            "matches_at_k": {}
        }

        # Check each field at each k level
        for check_k in range(1, min(k, len(pred_list)) + 1):
            field_matches = {}
            all_match = True

            for field in fields:
                gt_val = normalize_string(gt.get(field, ""))
                matched = False

                # Check if GT matches any of top-k predictions
                for p in pred_list[:check_k]:
                    pred_val = normalize_string(p.get(field, "") if isinstance(p, dict) else "")
                    fuzzy_score = fuzz.ratio(pred_val, gt_val) if pred_val and gt_val else 0
                    if fuzzy_score >= fuzzy_threshold:
                        matched = True
                        break

                field_matches[field] = matched
                if matched:
                    topk_correct[check_k][field] += 1
                else:
                    all_match = False

            result["matches_at_k"][check_k] = field_matches
            if all_match:
                topk_all_correct[check_k] += 1

        detailed_results.append(result)

    # Calculate metrics
    metrics = {
        "total_samples": total,
        "stage": stage,
        "k": k,
        "topk_accuracy": {},
        "topk_field_accuracy": {},
    }

    for check_k in range(1, k + 1):
        metrics["topk_accuracy"][f"top{check_k}"] = topk_all_correct[check_k] / total if total > 0 else 0
        metrics["topk_field_accuracy"][f"top{check_k}"] = {
            f: topk_correct[check_k][f] / total if total > 0 else 0 for f in fields
        }

    metrics["detailed_results"] = detailed_results

    return metrics

###Results Parsing and Analysis

In [ ]:
def load_predictions(output_dir: str) -> Tuple[List[Dict], List[Dict]]:
    """
    Load predictions and ground truth from LLaMA Factory output directory.
    """
    predictions_file = os.path.join(output_dir, "generated_predictions.jsonl")

    if not os.path.exists(predictions_file):
        raise FileNotFoundError(f"Predictions file not found: {predictions_file}")

    predictions = []
    ground_truth = []

    with open(predictions_file, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line.strip())

            # Parse prediction
            pred_text = entry.get("predict", "")
            pred_json = parse_json_response(pred_text)
            predictions.append(pred_json)

            # Parse ground truth (label)
            gt_text = entry.get("label", "")
            gt_json = parse_json_response(gt_text)
            ground_truth.append(gt_json)

    return predictions, ground_truth

In [ ]:
def evaluate_model(
    output_dir: str,
    stage: int,
    eval_mode: str = "top1",
    save_results: bool = True
) -> Dict[str, Any]:
    """
    Evaluate a model's predictions and calculate metrics.
    """
    print(f"\nEvaluating: {output_dir}")
    print(f"Mode: {eval_mode}")

    # Load predictions
    predictions, ground_truth = load_predictions(output_dir)
    print(f"Loaded {len(predictions)} predictions")

    # Calculate metrics based on eval_mode
    if eval_mode == "top1":
        metrics = calculate_accuracy(predictions, ground_truth, stage)

        # Print summary
        print(f"Stage {stage} Results (Top-1)")
        print(f"Total samples: {metrics['total_samples']}")
        print(f"Overall accuracy: {metrics['overall_accuracy']:.2%}")
        print("\nPer-field fuzzy accuracy:")
        for field, acc in metrics['fuzzy_accuracy'].items():
            print(f"  {field}: {acc:.2%}")
    else:
        k = EVAL_MODES[eval_mode]["k"]
        metrics = calculate_topk_accuracy(predictions, ground_truth, stage, k=k)

        # Print summary
        print(f"Stage {stage} Results (Top-{k})")
        print(f"Total samples: {metrics['total_samples']}")
        print("\nTop-k accuracy (all fields correct):")
        for key, acc in metrics['topk_accuracy'].items():
            print(f"  {key}: {acc:.2%}")

    if stage == 4 and "coordinate_metrics" in metrics:
        print("\nCoordinate metrics:")
        print(f"  Mean distance: {metrics['coordinate_metrics']['mean_distance_km']:.1f} km")
        print(f"  Within 100km: {metrics['coordinate_metrics']['within_100km']:.2%}")

    # Save results
    if save_results:
        summary_metrics = {k: v for k, v in metrics.items() if k != "detailed_results"}
        results_file = os.path.join(output_dir, f"evaluation_metrics_{eval_mode}.json")
        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(summary_metrics, f, indent=2, ensure_ascii=False)
        print(f"\nResults saved to: {results_file}")

    return metrics

In [ ]:
def evaluate_all_models(
    config: EvalConfig,
    model_name: str,
    gpu: str,
    dataset_sizes: List[str] = None,
    stages: List[int] = None,
    include_base: bool = True,
    eval_mode: str = "top1"
) -> Dict[str, Dict]:
    """
    Evaluate all models for a given configuration.
    """
    if dataset_sizes is None:
        dataset_sizes = SIZES
    if stages is None:
        stages = STAGES

    all_results = {}

    # Evaluate base model
    if include_base:
        print(f"Evaluating Base Model: {model_name} ({eval_mode})")

        for stage in stages:
            output_dir = os.path.join(
                config.results_base,
                f"{model_name}/{eval_mode}/base_model/stage_{stage}"
            )

            if os.path.exists(os.path.join(output_dir, "generated_predictions.jsonl")):
                key = f"{model_name}_base_s{stage}"
                all_results[key] = evaluate_model(output_dir, stage, eval_mode=eval_mode)
            else:
                print(f"Skipping {model_name} base stage {stage} (no predictions found)")

    print(f"Evaluating Fine-Tuned models: {model_name} ({gpu}, {eval_mode})")

    for size in dataset_sizes:
        for stage in stages:
            output_dir = os.path.join(
                config.results_base,
                f"{model_name}/{eval_mode}/{gpu}/{size}/stage_{stage}"
            )

            if os.path.exists(os.path.join(output_dir, "generated_predictions.jsonl")):
                key = f"{model_name}_{gpu}_{size}_s{stage}"
                all_results[key] = evaluate_model(output_dir, stage, eval_mode=eval_mode)
            else:
                print(f"Skipping {model_name} {gpu} {size} stage {stage} (no predictions found)")

    return all_results

###Main Execution Functions

In [ ]:
def quick_evaluate(
    config: EvalConfig,
    model_name: str = "qwen3vl-4b",
    gpu: str = "l4",
    dataset_size: str = "10000",
    stage: int = 1,
    eval_mode: str = "top1"
) -> Dict[str, Any]:
    """
    Quick evaluation of a single model configuration.

    """
    # Generate YAML
    yaml_path = generate_finetuned_model_eval_yaml(
        config, model_name, gpu, dataset_size, stage, eval_mode=eval_mode
    )

    # Run evaluation
    success = run_evaluation(yaml_path, config.llama_factory_path)

    if not success:
        print("Evaluation failed!")
        return {}

    # Calculate metrics
    output_dir = os.path.join(
        config.results_base,
        f"{model_name}/{eval_mode}/{gpu}/{dataset_size}/stage_{stage}"
    )

    return evaluate_model(output_dir, stage, eval_mode=eval_mode)

###Extract Images

In [ ]:
import gzip
import shutil
import tarfile
from tqdm import tqdm

In [ ]:
def extract_archives(archive_dir: Path, output_dir: Path, label: str) -> List[str]:
    """
    Extract tar.gz archives from Google Drive to local Colab storage.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    archives = sorted(list(archive_dir.glob("*.tar.gz")))

    if not archives:
        print(f"No archives found in {archive_dir}")
        return []

    print(f"Extracting {len(archives)} {label} archives to {output_dir}")

    failed = []
    for archive in tqdm(archives, desc=label):
        local_path = output_dir / archive.name
        try:

            # Copy archive to local storage first (faster extraction), extract,
            # and remove the copied archive
            shutil.copy(archive, local_path)

            with tarfile.open(local_path, "r:gz") as tar:
                tar.extractall(path=output_dir, filter='data')

            os.remove(local_path)

        except (EOFError, tarfile.ReadError, OSError, gzip.BadGzipFile) as e:
            print(f"\nFailed: {archive.name} - {e}")
            failed.append(archive.name)
            if local_path.exists():
                os.remove(local_path)

    print(f"Extracted {len(archives) - len(failed)}/{len(archives)} successfully")
    return failed

In [ ]:
# --- Extract image archives ---
DRIVE_ARCHIVE_DIR = Path("/content/drive/MyDrive/NLP/Thesis/image-archives/train")
TEST_ARCHIVE_DIR = Path("/content/drive/MyDrive/NLP/Thesis/image-archives/test")
LOCAL_TRAIN_DIR = Path("/content/images/train")
LOCAL_TEST_DIR = Path("/content/images/test")

if DRIVE_ARCHIVE_DIR.exists():
    failed_train = extract_archives(DRIVE_ARCHIVE_DIR, LOCAL_TRAIN_DIR, "train")
else:
    print("Train archive directory not found")

if TEST_ARCHIVE_DIR.exists():
    failed_test = extract_archives(TEST_ARCHIVE_DIR, LOCAL_TEST_DIR, "test")
else:
    print("Test archive directory not found")

Extracting 86 train archives to /content/images/train


train: 100%|██████████| 86/86 [14:07<00:00,  9.86s/it]


Extracted 86/86 successfully
Extracting 2 test archives to /content/images/test


test: 100%|██████████| 2/2 [00:19<00:00,  9.91s/it]

Extracted 2/2 successfully


###Evaluation Workflow

In [ ]:
config = EvalConfig()

Needed for nlg evaluation

In [ ]:
!pip install rouge-chinese rouge nltk jieba

####Merge Dataset Info

In [ ]:
# Merge dataset info into LLaMA Factory
merge_dataset_info(
    llama_factory_path="/content/LLaMA-Factory",
    base_drive_path="/content/drive/MyDrive/NLP/Thesis",
    test_subset_sizes=[10000]
)

Merged 224 dataset entries into /content/LLaMA-Factory/data/dataset_info.json
  Qwen entries: 112
  Gemma entries: 112


####Batch Evaluation

#####qwen

In [ ]:
# Generate all YAMLs and run batch evaluation

yaml_paths = generate_all_eval_yamls(
    config,
    model_name="qwen3vl-4b",
    gpu="l4",
    dataset_sizes=["1000", "10000"],
    stages=[1, 2, 3, 4],
    test_subset_size=10000,
    eval_mode="top1",
    include_base=True
)

# Run the evaluations
results = run_batch_evaluation(yaml_paths, config.llama_factory_path)

#####gemma

In [ ]:
# %pip install --upgrade huggingface_hub
from huggingface_hub import login as hf_login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
hf_login(token = hf_token)

In [ ]:
# Generate YAMLs
yaml_paths = generate_all_eval_yamls(
    config,
    model_name="gemma-3-4b-it",
    gpu="l4",
    dataset_sizes=["1000", "10000"],
    stages=[1, 2, 3, 4],
    test_subset_size=10000,
    eval_mode="top1",
    include_base=True
)

# Run all evaluations
results = run_batch_evaluation(yaml_paths, config.llama_factory_path)


--- Base Model YAMLs (gemma-3-4b-it) ---
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_base_eval_s1.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_base_eval_s2.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_base_eval_s3.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_base_eval_s4.yaml

--- Fine-tuned Model YAMLs (gemma-3-4b-it, l4) ---
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_l4_1000_eval_s1.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_l4_1000_eval_s2.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_l4_1000_eval_s3.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_l4_1000_eval_s4.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/gemma-3-4b-it_l4_10000_eval_s1.yaml
Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eva

####Evaluation Metrics

In [ ]:
config = EvalConfig()

######qwen

In [ ]:
# Calculate metrics from existing predictions
all_metrics = evaluate_all_models(
    config,
    model_name="qwen3vl-4b",
    gpu="l4",
    dataset_sizes=["1000", "10000"],
    stages=[1, 2, 3, 4],
    include_base=True,
)


EVALUATING BASE MODEL: qwen3vl-4b (top1)

Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_1
Mode: top1
Loaded 10000 predictions

--- Stage 1 Results (Top-1) ---
Total samples: 10000
Overall accuracy: 62.09%

Per-field fuzzy accuracy:
  country: 62.09%

Results saved to: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_1/evaluation_metrics_top1.json

Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_2
Mode: top1
Loaded 10000 predictions

--- Stage 2 Results (Top-1) ---
Total samples: 10000
Overall accuracy: 3.96%

Per-field fuzzy accuracy:
  region: 4.15%
  country: 52.61%

Results saved to: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_2/evaluation_metrics_top1.json

Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/to

#####gemma

In [ ]:
all_metrics = evaluate_all_models(
    config,
    model_name="gemma-3-4b-it",
    gpu="l4",
    dataset_sizes=["1000", "10000"],
    stages=[1, 2, 3, 4],
    include_base=True,
)

####Evaluate Base Model Only

In [ ]:
# Merge dataset info
merge_dataset_info(
    llama_factory_path="/content/LLaMA-Factory",
    base_drive_path="/content/drive/MyDrive/NLP/Thesis",
    test_subset_sizes=[10000]
)

Merged 112 dataset entries into /content/LLaMA-Factory/data/dataset_info.json
  Qwen entries: 56
  Gemma entries: 56


In [ ]:
%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [ ]:
stage = 4
yaml_path = generate_base_model_eval_yaml(
    config,
    "qwen3vl-4b",
    stage,
    test_dataset=f"hotel50k_qwen_test_subset_10000_s{stage}",
    batch_size=32,  # low batch size
    eval_mode="top1"
)
run_evaluation(yaml_path, config.llama_factory_path)

Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/qwen3vl-4b_base_eval_s4.yaml

Running evaluation: qwen3vl-4b_base_eval_s4.yaml

2026-01-28 10:08:43.977352: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-28 10:08:43.995591: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769594924.017203    8168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769594924.023804    8168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0

True

In [ ]:
# def generate_finetuned_model_eval_yaml(
#     config: EvalConfig,
#     model_name: str = "qwen3vl-4b",
#     gpu: str = "l4",
#     dataset_size: str = "10000",
#     stage: int = 1,
#     test_dataset: str = None,
#     batch_size: int = None,
#     eval_mode: str = "top1"
# ) -> str

In [ ]:
stage = 3
yaml_path = generate_finetuned_model_eval_yaml(
    config,
    "qwen3vl-4b",
    "l4",
    "10000",
    stage,
    test_dataset = f"hotel50k_qwen_test_subset_10000_s{stage}",
    batch_size = 32,
    eval_mode = "top1"
)
run_evaluation(yaml_path, config.llama_factory_path)

Generated: /content/drive/MyDrive/NLP/Thesis/yaml-eval-files/qwen3vl-4b_l4_10000_eval_s3.yaml

Running evaluation: qwen3vl-4b_l4_10000_eval_s3.yaml

2026-01-28 13:27:59.851395: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-28 13:27:59.871705: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769606879.894283   57377 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769606879.901089   57377 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regis

True

In [ ]:
model = "qwen3vl-4b"
model_type = "l4"
dataset_size = "10000"
output_dir = f"{config.results_base}/{model}/top1/{model_type}/{dataset_size}/stage_3"
if os.path.exists(f"{output_dir}/generated_predictions.jsonl"):
    metrics = evaluate_model(output_dir, stage=3, eval_mode="top1")


Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/l4/10000/stage_3
Mode: top1
Loaded 10000 predictions

--- Stage 3 Results (Top-1) ---
Total samples: 10000
Overall accuracy: 6.20%

Per-field fuzzy accuracy:
  city: 6.27%
  region: 16.41%
  country: 62.13%

Results saved to: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/l4/10000/stage_3/evaluation_metrics_top1.json


###Qwen Fix

In [ ]:
%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [ ]:
# Merge dataset info
merge_dataset_info(
    llama_factory_path="/content/LLaMA-Factory",
    base_drive_path="/content/drive/MyDrive/NLP/Thesis",
    test_subset_sizes=[10000]
)

# Generate evaluation YAMLs
yaml_paths = generate_all_eval_yamls(
    config,
    model_name="qwen3vl-4b",
    gpu="l4",
    dataset_sizes=["1000", "10000"],
    stages=[1, 2, 3, 4],
    test_subset_size=10000,
    eval_mode="top1",
    include_base=True
)

# Run evaluations
results = run_batch_evaluation(yaml_paths, config.llama_factory_path)

Streaming output truncated to the last 5000 lines.
100%|██████████| 313/313 [19:05<00:00,  2.90s/it]Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 0.870 seconds.
DEBUG:jieba:Loading model cost 0.870 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.

100%|██████████| 313/313 [19:11<00:00,  3.68s/it]
***** predict metrics *****
  predict_bleu-4                 =    80.1224
  predict_model_preparation_time =     0.0091
  predict_rouge-1                =    92.2482
  predict_rouge-2                =    89.6116
  predict_rouge-l                =    94.6988
  predict_runtime                = 0:19:14.95
  predict_samples_per_second     =      8.658
  predict_steps_per_second       =      0.271
[INFO|2026-01-27 20:34:01] llamafactory.train.sft.trainer:143 >> 

In [ ]:
all_metrics = evaluate_all_models(
    config,
    model_name="qwen3vl-4b",
    gpu="l4",
    dataset_sizes=["1000", "10000"],
    stages=[1, 2, 3, 4],
    include_base=True,
)


EVALUATING BASE MODEL: qwen3vl-4b (top1)

Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_1
Mode: top1
Loaded 10000 predictions

--- Stage 1 Results (Top-1) ---
Total samples: 10000
Overall accuracy: 62.09%

Per-field fuzzy accuracy:
  country: 62.09%

Results saved to: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_1/evaluation_metrics_top1.json

Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_2
Mode: top1
Loaded 10000 predictions

--- Stage 2 Results (Top-1) ---
Total samples: 10000
Overall accuracy: 3.96%

Per-field fuzzy accuracy:
  region: 4.15%
  country: 52.61%

Results saved to: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/top1/base_model/stage_2/evaluation_metrics_top1.json

Evaluating: /content/drive/MyDrive/NLP/Thesis/llama_factory_evaluation_results/qwen3vl-4b/to

###Stop Execution

In [ ]:
from google.colab import runtime
runtime.unassign()